# 1.2 Scrapper Robusto com Ollama
----
## Scraper Inteligente com Ollama e Llama 3.2

### Visão Geral
> Este notebook apresenta a construção de um pipeline simples de Inteligência Artificial Generativa utilizando modelos executados localmente por meio do Ollama.
>
> O projeto combina técnicas de Web Scraping, Engenharia de Prompt e Large Language Models (LLMs) para transformar o conteúdo bruto de páginas web em resumos estruturados e contextualizados.
>
> Ao longo do fluxo, uma página web é processada, seu conteúdo textual é extraído e limpo, e em seguida enviado ao modelo Llama 3.2, responsável por analisar as informações e gerar um resumo em linguagem natural.

### Objetivos do Projeto
> Este experimento tem como principais objetivos:

- Executar um modelo de linguagem localmente utilizando Ollama;
- Explorar conceitos básicos de Web Scraping com Python;
- Aplicar estratégias de Prompt Engineering;
- Utilizar a interface compatível com a API da OpenAI disponibilizada pelo Ollama;
- Gerar resumos automáticos a partir de websites;
- Reproduzir uma arquitetura semelhante à apresentada no curso do Ed Donner sem depender de serviços pagos.

### Arquitetura da Solução

```text
Website
   ↓
Web Scraping
   ↓
Limpeza do HTML
   ↓
Construção dos Prompts
   ↓
Llama 3.2 (Ollama)
   ↓
Resumo Gerado
```

## 1.2.1 Configuração do Ambiente
----

### Objetivo
> Antes de utilizar um modelo local, precisamos configurar o ambiente Python e estabelecer a comunicação com o servidor Ollama.

Nesta etapa são importadas todas as bibliotecas necessárias para:

- Realizar requisições HTTP;
- Processar conteúdo HTML;
- Exibir resultados formatados em Markdown;
- Interagir com o Ollama através da interface compatível com a API OpenAI.

### Bibliotecas Utilizadas

**1. OpenAI SDK**
>Embora este projeto não utilize os serviços da OpenAI, o SDK é aproveitado por sua compatibilidade com a API exposta pelo Ollama.
>
>Dessa forma, podemos utilizar a mesma estrutura de chamadas empregada em aplicações baseadas na OpenAI.

<br>

**2. Requests**
> Biblioteca responsável pela comunicação HTTP.
>
> Será utilizada para acessar e baixar o conteúdo dos websites analisados.

<br>

**3. BeautifulSoup**
> Ferramenta de parsing HTML amplamente utilizada em projetos de Web Scraping.
>
> Permite navegar, filtrar e extrair informações de documentos HTML de forma simples.

<br>

**4. IPython Display**
> Utilizado apenas para exibir o resultado final formatado em Markdown dentro do notebook.

### Configuração do Cliente Ollama

O Ollama disponibiliza uma API local que pode ser acessada através do endereço:
```
http://localhost:11434/v1
````

Ao configurar o SDK para apontar para essa URL, todas as chamadas realizadas serão processadas localmente pelo modelo instalado na máquina.

Como o Ollama não exige autenticação por chave de API, um valor fictício é informado apenas para atender aos requisitos do SDK.

### Célula 1: Importações e Configuração

In [ ]:
# Célula 1: Importações e Configuração

from openai import OpenAI
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

# Configuração do cliente OpenAI apontando para o Ollama local
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="not-needed"  # O Ollama não exige chave real, mas o SDK exige o parâmetro
)

print("Ambiente configurado com sucesso!")

## 1.2.2 Coleta e Limpeza dos Dados
----

### Objetivo
> Modelos de linguagem dependem diretamente da qualidade dos dados fornecidos.

Uma página web contém diversos elementos que não agregam valor à análise, como:

- Scripts JavaScript;
- Folhas de estilo;
- Menus de navegação;
- Rodapés;
- Campos de formulário;
- Imagens.

O objetivo desta etapa é transformar uma página HTML complexa em texto limpo e relevante para o modelo.

### Estratégia de Extração

O processo executa as seguintes etapas:

**1. Download da Página**
> Uma requisição HTTP é enviada para o endereço informado.
>
> Para reduzir bloqueios por parte dos servidores, é utilizado um User-Agent semelhante ao de um navegador real.

<br>

**2. Validação da Resposta**
> Caso o servidor retorne algum erro (404, 500, timeout, etc.), a exceção é tratada e uma mensagem amigável é retornada.
>
> Essa prática torna o pipeline mais robusto e adequado para ambientes produtivos.

<br>

**3. Parsing do HTML**
> Após o download, o conteúdo é processado pelo BeautifulSoup.
>
> Nesse momento o HTML é convertido para uma estrutura navegável em Python.

<br>

**4. Limpeza do Conteúdo**
> Elementos considerados irrelevantes para a análise são removidos antes da geração do contexto.
>
> Essa etapa reduz ruído e melhora significativamente a qualidade das respostas produzidas pelo modelo.

<br>

**5. Controle de Tamanho**
> Como modelos possuem limites de contexto, o texto final é truncado para um tamanho seguro.
>
> Essa estratégia evita erros de processamento e reduz o consumo de recursos computacionais.

### Célula 2: Função de Web Scraping Robusta

In [ ]:
# Célula 2: Função de Web Scraping Robusta

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def fetch_website_contents(url: str, max_chars: int = 4000) -> str:
    """
    Realiza o scraping de uma URL, remove elementos irrelevantes 
    e retorna o texto limpo truncado para o limite seguro do modelo.
    """
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status() # Lança exceção para códigos de erro HTTP (ex: 404, 500)
    except requests.exceptions.RequestException as e:
        return f"Erro ao acessar o site: {e}"

    soup = BeautifulSoup(response.content, "html.parser")

    # Extrai o título ou define um padrão
    title = soup.title.string.strip() if soup.title and soup.title.string else "Sem título encontrado"

    if soup.body:
        # Remove elementos que poluem o contexto da IA
        for irrelevant in soup.body(["script", "style", "img", "input", "nav", "footer"]):
            irrelevant.decompose()

        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""

    # Junta o título e o corpo, limitando os caracteres para evitar estouro de contexto
    combined_content = f"Título: {title}\n\nConteúdo:\n{text}"
    return combined_content[:max_chars]

## 1.2.3 Engenharia de Prompt
----
### Objetivo
> Após a coleta dos dados, precisamos orientar o modelo sobre como ele deve interpretar e responder às informações recebidas.
>
> Esse processo é conhecido como Prompt Engineering.

**1. System Prompt**
O System Prompt define o comportamento do modelo.

Neste projeto ele estabelece que a IA deve atuar como:
> Um analista sênior de mídias e tecnologia.

Além disso, define que a resposta deve ser produzida em formato Markdown.

Essa abordagem ajuda a manter consistência entre diferentes execuções.

<br>

**2. User Prompt**

O User Prompt representa a tarefa específica.

Neste caso o modelo recebe duas responsabilidades:

- Identificar as principais notícias presentes no conteúdo;
- Produzir uma breve opinião utilizando tom satírico ou irônico.

Essa separação entre System Prompt e User Prompt é considerada uma boa prática em aplicações baseadas em LLMs.

<br>

**3. Estrutura de Messages**

Os modelos compatíveis com a API OpenAI trabalham utilizando uma estrutura baseada em mensagens.

Cada mensagem possui:

- Um papel (role);
- Um conteúdo (content).

Exemplo:
```text
System → Define comportamento
User → Define tarefa
Assistant → Resposta gerada
```

### Célula 3: Engenharia de Prompts e Messages

In [ ]:
# Célula 3: Engenharia de Prompts e Messages

system_prompt = (
    "Você é um analista sênior de mídias e tecnologia. "
    "Sua tarefa é analisar o conteúdo extraído de um site e entregá-lo formatado em Markdown."
)

def messages_for(website_content: str) -> list:
    """
    Constrói a estrutura de mensagens no padrão esperado pela API da OpenAI/Ollama.
    """
    user_prompt = (
        "Aqui está o conteúdo extraído de um site de notícias. "
        "Forneça as principais notícias com uma breve opinião em forma de sátira ou ironia:\n\n"
        f"{website_content}"
    )
    
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

## 4. Pipeline de Inferência
----

### Objetivo
>Nesta etapa conectamos todas as partes do sistema.

O pipeline completo executa:

- Coleta do website;
- Limpeza do conteúdo;
- Construção dos prompts;
- Envio para o modelo;
- Recebimento da resposta;
- Exibição do resultado formatado.

### Chamada ao Modelo

O método:

`ollama.chat.completions.create()`

envia as mensagens para o modelo Llama 3.2 em execução local.

O retorno contém diversas informações, incluindo:

- Metadados da resposta;
- Tokens utilizados;
- Mensagem gerada pelo modelo.

Neste projeto extraímos apenas o conteúdo textual retornado.

### Controle de Criatividade

O parâmetro:

`temperature=0.7`

controla o nível de aleatoriedade das respostas.

Em linhas gerais:

| Temperatura | Comportamento |
| :--- | :--- |
| 0.0 | Mais determinístico |
| 0.3 | Conservador |
| 0.7 | Equilibrado |
| 1.0+ | Mais criativo |

Como o projeto solicita comentários satíricos, uma temperatura intermediária produz resultados mais interessantes.

### Teste Prático
Cenário:

Como exemplo, será utilizada uma seção de notícias sobre Inteligência Artificial.

O conteúdo da página será coletado automaticamente e enviado ao modelo Llama 3.2 para análise.

Ao final, o notebook exibirá um resumo estruturado contendo:

- Principais manchetes;
- Destaques do conteúdo;
- Comentários satíricos gerados pela IA.

### Célula 4: Pipeline de Resumo e Execução

In [ ]:
# Célula 4: Pipeline de Resumo e Execução

def summarize(url: str, model_name: str = "llama3.2") -> str:
    print(f"🔄 Coletando dados de: {url}...")
    website_content = fetch_website_contents(url)
    
    if "Erro ao acessar" in website_content:
        return website_content

    print(f"🤖 Gerando resumo com o modelo {model_name}...")
    try:
        response = ollama.chat.completions.create(
            model=model_name,
            messages=messages_for(website_content),
            temperature=0.7 # Adiciona um pouco de criatividade para a sátira
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Erro ao comunicar com o Ollama: {e}"

# Testando o pipeline completo
url_alvo = "https://www.cnnbrasil.com.br/ia/"
summary = summarize(url_alvo)

display(Markdown(summary))

## Conclusão

Este projeto demonstra como construir uma aplicação simples de Inteligência Artificial Generativa utilizando apenas ferramentas de código aberto e execução local.

A combinação entre Web Scraping, Prompt Engineering e LLMs oferece uma base sólida para o desenvolvimento de soluções capazes de transformar informações não estruturadas em conhecimento útil, sem depender de APIs comerciais.

Além do valor educacional, a arquitetura apresentada pode servir como ponto de partida para sistemas mais complexos de monitoramento de informação, análise de conteúdo e automação baseada em IA.